# Code to do Autoencoder


Import libraries

In [ ]:
from torch import nn
import torch
import numpy as np
from Autoencoder import AutoEncoder

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Example dataset

Here loading the dataset MNIST. To train on another data load the real data instead.

In [42]:
from torchvision.datasets import MNIST
from torch.utils.data import ConcatDataset
from torchvision import transforms

# Transform into a tensor
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,)),
                                 ])
# Loading Training Data and Test Data
trainset = MNIST('./', download=True,
                 train = True,
                 transform=transform)
testset = MNIST(root='./', download=True,
                train=False,
                transform=transform)
data_set = ConcatDataset([trainset, testset])

dataloader = torch.utils.data.DataLoader(data_set,
                                         batch_size=256,
                                         shuffle=True,
                                         num_workers=10)




In [44]:
X_train = trainset.data.numpy().reshape(-1, 28*28)
X_test = testset.data.numpy().reshape(-1, 28*28)
X_test.shape

y_train = np.array(trainset.targets)
y_test = np.array(testset.targets)

y = np.concatenate([y_train, y_test])
X = np.concatenate([X_train, X_test])
X.shape

# Save the data
np.savez_compressed('MNIST_data.npz', X=X, y=y)

#### Pre-Training the Model

Define the model, optimizer, learning rate and loss function

In [34]:
import torch.optim.lr_scheduler as lr_scheduler

n_input_features = 784 # 28x28 images flattened
model = AutoEncoder(input_dim=n_input_features,
                          hidden_layers=[500, 500, 2000, 10],
                          dropout_rate=0.2
                          ).to(device)
# Activate training mode
model.train()

# Learning Rate
lr = 0.1

# Use Stochastic Gradient Descent optimizer
optimizer = torch.optim.SGD(lr=lr,
                            momentum=0.9,
                            params = model.parameters())
# Use Mean Squared Error Loss as loss function
loss_fn = nn.MSELoss()

# Use learning rate decay - reduce learning rate as training progresses
scheduler = lr_scheduler.StepLR(optimizer,
                                step_size=100,
                                gamma=0.1)



Do the pretraining

In [35]:
n_epochs = 200
eval_every = 10
best_loss = np.infty

for epoch in range(n_epochs):
    losses = []

    for x_batch, y_batch in dataloader:
        # Reset the gradients (only necessary for PyTorch)
        optimizer.zero_grad()  # Zero the gradients

        # 0. Flatten the image from 28x28 to 784
        x_batch = x_batch.to(device)
        x_batch = x_batch.view(x_batch.shape[0], -1)  # Flatten the images

        # 1. Apply the autoencoder model
        output = model(x_batch)[1]

        # 2. Calculate the reconstruction loss
        loss = loss_fn(output, x_batch)
        losses.append(loss.item())

        # 3. Backpropagate the loss 
        loss.backward()

        # 4. Update the model parameters (weights)
        optimizer.step()


    # Mean loss of the batches in this epoch
    mean_loss = np.round(np.mean(losses), 5)

    if (epoch + 1) % eval_every == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss}")

    scheduler.step()

Epoch 10/200, Loss: 0.15413
Epoch 20/200, Loss: 0.12074
Epoch 30/200, Loss: 0.10639
Epoch 40/200, Loss: 0.09953
Epoch 50/200, Loss: 0.09501
Epoch 60/200, Loss: 0.09164
Epoch 70/200, Loss: 0.08904
Epoch 80/200, Loss: 0.08695
Epoch 90/200, Loss: 0.08519
Epoch 100/200, Loss: 0.08361
Epoch 110/200, Loss: 0.08293
Epoch 120/200, Loss: 0.08278
Epoch 130/200, Loss: 0.08258
Epoch 140/200, Loss: 0.08247
Epoch 150/200, Loss: 0.08229
Epoch 160/200, Loss: 0.08218
Epoch 170/200, Loss: 0.08209
Epoch 180/200, Loss: 0.08197
Epoch 190/200, Loss: 0.08183
Epoch 200/200, Loss: 0.08166


Store the model

In [37]:
torch.save(model, './autoencoder_model')
model = torch.load('./autoencoder_model', weights_only=False)

#### Fine-tuning the AutoEncoder

In [39]:
model = torch.load('./autoencoder_model', weights_only=False)

model.eval()

lr = 0.1
optimizer = torch.optim.SGD(lr=lr,
                            momentum=0.9,
                            params = model.parameters())  

n_epochs = 50
eval_every = 10
best_loss = np.infty

for epoch in range(n_epochs):
    for x_batch, y_batch in dataloader:
        # Reset the gradients (only necessary for PyTorch)
        optimizer.zero_grad()  # Zero the gradients

        # 0. Flatten the image from 28x28 to 784
        x_batch = x_batch.to(device)
        x_batch = x_batch.view(x_batch.shape[0], -1)  # Flatten the images

        # 1. Apply the autoencoder model
        output = model(x_batch)[1]

        # 2. Calculate the reconstruction loss
        loss = loss_fn(output, x_batch)
        losses.append(loss.item())

        # 3. Backpropagate the loss 
        loss.backward()

        # 4. Update the model parameters (weights)
        optimizer.step()


    # Mean loss of the batches in this epoch
    mean_loss = np.round(np.mean(losses), 5)

    if (epoch + 1) % eval_every == 0:
        print(f"[Fine-tuning] Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss}")

    scheduler.step()

[Fine-tuning] Epoch 10/50, Loss: 0.06094
[Fine-tuning] Epoch 20/50, Loss: 0.0582
[Fine-tuning] Epoch 30/50, Loss: 0.05645
[Fine-tuning] Epoch 40/50, Loss: 0.05509
[Fine-tuning] Epoch 50/50, Loss: 0.05396


In [40]:
torch.save(model, './autoencoder_model_finetuned')
model = torch.load('./autoencoder_model_finetuned', weights_only=False)